# Demo on mobility-CIR-CFR

- This code demonstrates how channel frequency response (CFR) and channel impulse response (CIR) changes as the transceivers move.
- The captured CFR and CIR can be animated.

In [15]:
import sionna.rt

import time
import os
import re

import cv2
from PIL import Image

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

no_preview = True # Toggle to False to use the preview widget

# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, RadioMapSolver, subcarrier_frequencies, ITURadioMaterial, SceneObject

import scipy.signal
import scipy.linalg

In [16]:
# scene = load_scene("../assets/ssm-scene/ssm-scene-v2_3.xml")
scene = load_scene(sionna.rt.scene.simple_street_canyon)

scene.frequency = 5e9

scene.tx_array = PlanarArray(num_rows=4,
                             num_cols=4,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

scene.rx_array = PlanarArray(num_rows=8,
                             num_cols=8,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="VH")

# Create transmitters.
tx1 = Transmitter(name="tx1", # Node 24.
                  # position=[3,-57,0.75],
                  position=[0,0,0.75],
                  velocity=[0, 10, 0])

rx1 = Receiver(name="rx1", # BS position 4.
               # position=[-5, -13, 20],
               position=[0, 0, 20],
               look_at=[0, 0, 0])

scene.add(tx1)
scene.add(rx1)

# scene.preview();

In [17]:
# 15 kHz subcarrier spacing.
# 264 subcarriers (Half of NR standards).
# 14 OFDM symbols per slot.
subcarrier_spacing = 15e3
num_subcarriers = 264
frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)
num_ofdm_symbols = 14
num_paths = 5 # Desired number of paths of CIR reconstruction.

bandwidth = num_subcarriers * subcarrier_spacing
print(f"Total bandwidth: {bandwidth / 1e6} MHz")
print(f"Time resolution: {1e6/bandwidth:.3f} us")

p_solver = PathSolver()

Total bandwidth: 3.96 MHz
Time resolution: 0.253 us


In [18]:
paths = p_solver(scene=scene,
                 max_depth=5,
                 los=False,
                 specular_reflection=True,
                 diffuse_reflection=True,
                 refraction=True,
                 synthetic_array=True,
                 seed=1)

## Generate CFR-CIR output for slowly moving transceivers.

In [ ]:
num_time_steps = 100

for t in range(num_time_steps):
    scene.get("tx1").position += [0.1, 0, 0]
    print(scene.get("tx1").position)
    paths = p_solver(scene=scene,
                 max_depth=5,
                 los=False,
                 specular_reflection=True,
                 specular_reflection=True,
                 diffuse_reflection=True,
                 refraction=True,
                 synthetic_array=True,
                 seed=1)
    cfr = paths.cfr(frequencies=frequencies,
                    sampling_frequency=num_subcarriers * subcarrier_spacing,
                    num_time_steps=num_ofdm_symbols,
                    normalize_delays=False,
                    normalize=False,
                    out_type="numpy")

    cir = paths.cir(sampling_frequency=num_subcarriers * subcarrier_spacing, # Sionna demo used subcarrier spacing instead of the total bandwidth.
                    num_time_steps=num_ofdm_symbols,
                    normalize_delays=False,
                    out_type="numpy")
    cir_a = cir[0]
    cir_tau = cir[1]

    # Target Tx, Rx, antenna.
    target_rx = 0
    target_tx = 0
    target_rx_ant = 0
    target_tx_ant = 0
        
    target_cfr = cfr[target_rx, target_rx_ant, target_tx, target_tx_ant, :, :]
    
    cir_target_a = cir_a[target_rx, target_rx_ant, target_tx, target_tx_ant, :, :]

    if len(cir[1].shape) == 3:
        cir_target_tau = cir_tau[target_rx, target_tx, :]
    else:
        cir_target_tau = cir_tau[target_rx, target_rx_ant, target_tx, target_tx_ant, :]
    
    valid_indices = np.where(cir_target_tau >= 0)[0]
    clean_tau = cir_target_tau[valid_indices]
    num_valid_paths = len(clean_tau)
    cir_target_a = cir_target_a[valid_indices]
    cir_target_tau = cir_target_tau[valid_indices]
    cir_target_tau_us = cir_target_tau * 1e6
    
    # Calculate Magnitude in dB
    # We add a tiny epsilon (1e-16) to avoid log(0) errors
    cir_target_a_db = 20 * np.log10(np.abs(cir_target_a) + 1e-12)
    # Square to calculate power.
    target_cir_power_avg = np.mean(np.abs(cir_target_a) ** 2, axis=1)
    
    # Calculate Average PDP (averaged across all OFDM symbols).
    target_cir_pdp_db_avg = np.mean(cir_target_a_db, axis=1)
    target_cir_power_avg = np.mean(np.abs(cir_target_a) ** 2, axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].set_title('Original CFR')
    im1 = axes[0].pcolormesh(np.abs(target_cfr.transpose()))
    axes[0].set_xlabel('Time (OFDM symbols)')
    axes[0].set_ylabel('Frequency (Subcarriers)')
    fig.colorbar(im1, ax=axes[0], label='Magnitude')

    markerline, stemlines, baseline = axes[1].stem(cir_target_tau_us, target_cir_power_avg, basefmt=" ")

    plt.setp(markerline, 'markerfacecolor', 'r', 'markersize', 6)
    plt.setp(stemlines, 'color', 'k', 'linewidth', 1.5)
    
    axes[1].set_xlabel('Delay ($\\mu s$)')
    axes[1].set_ylabel('Average Magnitude (Linear)')
    axes[1].set_title('Average Power Delay Profile (PDP)')
    # ax2.set_ylim([-1e-11, 1e-11])
    axes[1].grid(True, linestyle='--', alpha=0.5)
    
    markerline, stemlines, baseline = axes[2].stem(cir_target_tau_us, target_cir_pdp_db_avg, basefmt=" ")

    plt.setp(markerline, 'markerfacecolor', 'r', 'markersize', 6)
    plt.setp(stemlines, 'color', 'k', 'linewidth', 1.5)
    
    axes[2].set_xlabel('Delay ($\\mu s$)')
    axes[2].set_ylabel('Average Magnitude (dB)')
    axes[2].set_title('Average Power Delay Profile (PDP)')
    # ax2.set_ylim([-1e-11, 1e-11])
    axes[2].grid(True, linestyle='--', alpha=0.5)
    
    # plt.show()
    os.makedirs("demo_figs", exist_ok=True)
    plt.savefig(f"demo_figs/h-{t}.png")
    plt.close()
    

[[0.1, 0, 0.75]]
[[0.2, 0, 0.75]]
[[0.3, 0, 0.75]]
[[0.4, 0, 0.75]]
[[0.5, 0, 0.75]]
[[0.6, 0, 0.75]]
[[0.7, 0, 0.75]]
[[0.8, 0, 0.75]]
[[0.9, 0, 0.75]]
[[1, 0, 0.75]]
[[1.1, 0, 0.75]]
[[1.2, 0, 0.75]]
[[1.3, 0, 0.75]]
[[1.4, 0, 0.75]]
[[1.5, 0, 0.75]]
[[1.6, 0, 0.75]]
[[1.7, 0, 0.75]]
[[1.8, 0, 0.75]]
[[1.9, 0, 0.75]]
[[2, 0, 0.75]]
[[2.1, 0, 0.75]]
[[2.2, 0, 0.75]]
[[2.3, 0, 0.75]]
[[2.4, 0, 0.75]]
[[2.5, 0, 0.75]]
[[2.6, 0, 0.75]]
[[2.7, 0, 0.75]]
[[2.8, 0, 0.75]]
[[2.9, 0, 0.75]]
[[3, 0, 0.75]]
[[3.1, 0, 0.75]]
[[3.2, 0, 0.75]]
[[3.3, 0, 0.75]]
[[3.4, 0, 0.75]]
[[3.5, 0, 0.75]]
[[3.6, 0, 0.75]]
[[3.7, 0, 0.75]]
[[3.8, 0, 0.75]]
[[3.9, 0, 0.75]]
[[4, 0, 0.75]]
[[4.1, 0, 0.75]]
[[4.2, 0, 0.75]]
[[4.3, 0, 0.75]]
[[4.4, 0, 0.75]]
[[4.5, 0, 0.75]]
[[4.6, 0, 0.75]]
[[4.7, 0, 0.75]]
[[4.8, 0, 0.75]]
[[4.9, 0, 0.75]]
[[5, 0, 0.75]]
[[5.1, 0, 0.75]]
[[5.2, 0, 0.75]]
[[5.3, 0, 0.75]]
[[5.4, 0, 0.75]]
[[5.5, 0, 0.75]]
[[5.6, 0, 0.75]]
[[5.7, 0, 0.75]]
[[5.8, 0, 0.75]]
[[5.9, 0, 0.75]]
[[6, 0,

In [20]:
def img2vid(img_dir, time_slot_length, header="snapshot"):
    """
    Change set of png images to an mp4 video.
    The directory should contain images labeled in integers (1, 2, ...).
    """
    video_name = os.path.join(img_dir, f"video-{header}.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    images = [img for img in os.listdir(img_dir) if img.endswith('png')]

    # Classify according to header.
    images = [img for img in images if img.startswith(header)]

    # Sort.
    # sorted_images = sorted(images, key=lambda fn: int(fn.split('.', 1)[0]))
    def get_number_from_filename(filename):
        match = re.search(fr'{header}-(\d+)\.png', filename)
        if match:
            return int(match.group(1))
        return 0 # Default value if pattern doesn't match.
    images.sort(key=get_number_from_filename)

    frame = cv2.imread(os.path.join(img_dir, images[0]))

    height, width, layers = frame.shape
    video = cv2.VideoWriter(filename=video_name, fourcc=fourcc, fps=1/time_slot_length, frameSize=(width,height))
    for image in images:
        print(image)
        video.write(cv2.imread(os.path.join(img_dir, image)))

    cv2.destroyAllWindows()
    video.release()

img2vid("demo_figs", 0.5, "h")

h-0.png
h-1.png
h-2.png
h-3.png
h-4.png
h-5.png
h-6.png
h-7.png
h-8.png
h-9.png
h-10.png
h-11.png
h-12.png
h-13.png
h-14.png
h-15.png
h-16.png
h-17.png
h-18.png
h-19.png
h-20.png
h-21.png
h-22.png
h-23.png
h-24.png
h-25.png
h-26.png
h-27.png
h-28.png
h-29.png
h-30.png
h-31.png
h-32.png
h-33.png
h-34.png
h-35.png
h-36.png
h-37.png
h-38.png
h-39.png
h-40.png
h-41.png
h-42.png
h-43.png
h-44.png
h-45.png
h-46.png
h-47.png
h-48.png
h-49.png
h-50.png
h-51.png
h-52.png
h-53.png
h-54.png
h-55.png
h-56.png
h-57.png
h-58.png
h-59.png
h-60.png
h-61.png
h-62.png
h-63.png
h-64.png
h-65.png
h-66.png
h-67.png
h-68.png
h-69.png
h-70.png
h-71.png
h-72.png
h-73.png
h-74.png
h-75.png
h-76.png
h-77.png
h-78.png
h-79.png
h-80.png
h-81.png
h-82.png
h-83.png
h-84.png
h-85.png
h-86.png
h-87.png
h-88.png
h-89.png
h-90.png
h-91.png
h-92.png
h-93.png
h-94.png
h-95.png
h-96.png
h-97.png
h-98.png
h-99.png


## Generate UL CIR-DL CIR for slowly moving transceivers.

In [21]:
# Helper function to process CIR data for a specific link.
def get_pdp_data(cir_tuple, rx_idx=0, rx_ant_idx=0, tx_idx=0, tx_ant_idx=0):
    """
    Extracts delays and computes Average Power Delay Profile (PDP) from Sionna CIR output.
    """
    a, tau = cir_tuple
    
    # Sionna Output Shapes (Standard):
    # a: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps]
    # tau: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths]
    
    # Extract specific link data.
    # Squeezing batch dimension (index 0).
    # Shape becomes: [num_paths, num_time_steps] for 'a' and [num_paths] for 'tau'.
    link_a = a[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :, :]
    
    if len(tau.shape) == 3:
        link_tau = tau[rx_idx, tx_idx, :]
    else:
        link_tau = tau[rx_idx, rx_ant_idx, tx_idx, tx_ant_idx, :]

    # Filter invalid paths (Sionna pads with -1).
    valid_mask = link_tau >= 0
    clean_tau = link_tau[valid_mask]
    clean_a = link_a[valid_mask, :]

    # Convert delay to microseconds.
    tau_us = clean_tau * 1e6

    # Calculate Power (Linear).
    # Average the instantaneous power (|h|^2) over the time steps (OFDM symbols).
    pdp_linear = np.mean(np.abs(clean_a)**2, axis=1)

    # Calculate Power (dB).
    # 10*log10(Power). Add epsilon to avoid log(0).
    pdp_db = 10 * np.log10(pdp_linear + 1e-16)

    return tau_us, pdp_linear, pdp_db


In [22]:
num_time_steps = 100

for t in range(num_time_steps):
    # Update position
    scene.get("tx1").position += [0.1, 0, 0]

    # Run Path Solver
    paths = p_solver(scene=scene,
                 max_depth=5,
                 los=False,
                 specular_reflection=True,
                 diffuse_reflection=True,
                 refraction=True,
                 synthetic_array=True,
                 seed=1)

    # Generate CIR for Uplink and Downlink
    ul_cir = paths.cir(sampling_frequency=num_subcarriers * subcarrier_spacing,
                       num_time_steps=num_ofdm_symbols,
                       normalize_delays=False,
                       out_type="numpy")
    
    dl_cir = paths.cir(sampling_frequency=num_subcarriers * subcarrier_spacing,
                       num_time_steps=num_ofdm_symbols,
                       normalize_delays=False,
                       out_type="numpy",
                       reverse_direction=True)

    ul_tau, ul_pwr_lin, ul_pwr_db = get_pdp_data(ul_cir)
    dl_tau, dl_pwr_lin, dl_pwr_db = get_pdp_data(dl_cir)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f"Channel Impulse Response (CIR) - Step {t}", fontsize=16)

    markerline, stemlines, baseline = axes[0, 0].stem(ul_tau, ul_pwr_db, basefmt=" ")
    plt.setp(markerline, 'markerfacecolor', 'b', 'markersize', 6)
    plt.setp(stemlines, 'color', 'b', 'linewidth', 1.5)
    axes[0, 0].set_title("Uplink CIR (dB)")
    axes[0, 0].set_ylabel("Power (dB)")
    axes[0, 0].grid(True, linestyle='--', alpha=0.5)

    markerline, stemlines, baseline = axes[0, 1].stem(ul_tau, ul_pwr_lin, basefmt=" ")
    plt.setp(markerline, 'markerfacecolor', 'b', 'markersize', 6)
    plt.setp(stemlines, 'color', 'b', 'linewidth', 1.5)
    axes[0, 1].set_title("Uplink CIR (Linear)")
    axes[0, 1].set_ylabel("Power (Linear)")
    axes[0, 1].grid(True, linestyle='--', alpha=0.5)

    markerline, stemlines, baseline = axes[1, 0].stem(dl_tau, dl_pwr_db, basefmt=" ")
    plt.setp(markerline, 'markerfacecolor', 'r', 'markersize', 6)
    plt.setp(stemlines, 'color', 'r', 'linewidth', 1.5)
    axes[1, 0].set_title("Downlink CIR (dB)")
    axes[1, 0].set_ylabel("Power (dB)")
    axes[1, 0].set_xlabel(r"Delay ($\mu s$)")
    axes[1, 0].grid(True, linestyle='--', alpha=0.5)

    markerline, stemlines, baseline = axes[1, 1].stem(dl_tau, dl_pwr_lin, basefmt=" ")
    plt.setp(markerline, 'markerfacecolor', 'r', 'markersize', 6)
    plt.setp(stemlines, 'color', 'r', 'linewidth', 1.5)
    axes[1, 1].set_title("Downlink CIR (Linear)")
    axes[1, 1].set_ylabel("Power (Linear)")
    axes[1, 1].set_xlabel(r"Delay ($\mu s$)")
    axes[1, 1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    
    os.makedirs("demo_figs2", exist_ok=True) 
    plt.savefig(f"demo_figs2/h-{t}.png")
    plt.close()

print("Complete.")

Complete.


In [23]:
def img2vid(img_dir, time_slot_length, header="snapshot"):
    """
    Change set of png images to an mp4 video.
    The directory should contain images labeled in integers (1, 2, ...).
    """
    video_name = os.path.join(img_dir, f"video-{header}.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    images = [img for img in os.listdir(img_dir) if img.endswith('png')]

    # Classify according to header.
    images = [img for img in images if img.startswith(header)]

    # Sort.
    # sorted_images = sorted(images, key=lambda fn: int(fn.split('.', 1)[0]))
    def get_number_from_filename(filename):
        match = re.search(fr'{header}-(\d+)\.png', filename)
        if match:
            return int(match.group(1))
        return 0 # Default value if pattern doesn't match.
    images.sort(key=get_number_from_filename)

    frame = cv2.imread(os.path.join(img_dir, images[0]))

    height, width, layers = frame.shape
    video = cv2.VideoWriter(filename=video_name, fourcc=fourcc, fps=1/time_slot_length, frameSize=(width,height))
    for image in images:
        print(image)
        video.write(cv2.imread(os.path.join(img_dir, image)))

    cv2.destroyAllWindows()
    video.release()

img2vid("demo_figs2", 0.5, "h")

h-0.png
h-1.png
h-2.png
h-3.png
h-4.png
h-5.png
h-6.png
h-7.png
h-8.png
h-9.png
h-10.png
h-11.png
h-12.png
h-13.png
h-14.png
h-15.png
h-16.png
h-17.png
h-18.png
h-19.png
h-20.png
h-21.png
h-22.png
h-23.png
h-24.png
h-25.png
h-26.png
h-27.png
h-28.png
h-29.png
h-30.png
h-31.png
h-32.png
h-33.png
h-34.png
h-35.png
h-36.png
h-37.png
h-38.png
h-39.png
h-40.png
h-41.png
h-42.png
h-43.png
h-44.png
h-45.png
h-46.png
h-47.png
h-48.png
h-49.png
h-50.png
h-51.png
h-52.png
h-53.png
h-54.png
h-55.png
h-56.png
h-57.png
h-58.png
h-59.png
h-60.png
h-61.png
h-62.png
h-63.png
h-64.png
h-65.png
h-66.png
h-67.png
h-68.png
h-69.png
h-70.png
h-71.png
h-72.png
h-73.png
h-74.png
h-75.png
h-76.png
h-77.png
h-78.png
h-79.png
h-80.png
h-81.png
h-82.png
h-83.png
h-84.png
h-85.png
h-86.png
h-87.png
h-88.png
h-89.png
h-90.png
h-91.png
h-92.png
h-93.png
h-94.png
h-95.png
h-96.png
h-97.png
h-98.png
h-99.png
